## API Call

In [1]:
import re
import unicodedata
import requests
import pandas as pd


def get_nih_awards(
    first_name,
    middle_name,
    last_name,
    email,
    rpp=25,
    name_aliases=None
):
    """
    Retrieve NIH RePORTER projects associated with a researcher.

    Parameters
    ----------
    first_name, middle_name, last_name : str
        Researcher's name.
    email : str
        Researcher's email, retained for identification.
    rpp : int, default=25
        Number of projects requested per API call (maximum 500).
    name_aliases : list of str, optional
        Additional first-name variants, e.g. ["Bill", "William"].
        Each alias is combined with the supplied last name.

    Returns
    -------
    pandas.DataFrame
        One row per returned project, with investigator and match
        information retained for manual verification.
    """

    url = "https://api.reporter.nih.gov/v2/projects/search"

    def clean(value):
        if pd.isna(value):
            return ""
        value = str(value).strip()
        return "" if value.lower() == "nan" else value

    def normalize(value):
        value = unicodedata.normalize("NFKD", clean(value))
        value = "".join(
            char for char in value
            if not unicodedata.combining(char)
        )
        return re.sub(r"[^a-z0-9]", "", value.casefold())

    first = clean(first_name)
    middle = clean(middle_name)
    last = clean(last_name)

    if not first or not last:
        raise ValueError("first_name and last_name are required.")

    if not isinstance(rpp, int) or isinstance(rpp, bool) or rpp < 1:
        raise ValueError("rpp must be a positive integer.")

    limit = min(rpp, 500)


    # Build name variants for API retrieval.
    # The API's first/last-name search supports partial-name matching.
    search_names = [{
        "first_name": first,
        "last_name": last
    }]

    if middle:
        search_names.append({
            "first_name": first,
            "middle_name": middle,
            "last_name": last
        })

    for alias in name_aliases or []:
        alias = clean(alias)
        if alias:
            search_names.append({
                "first_name": alias,
                "last_name": last
            })

    # Remove duplicate search criteria.
    unique_names = []
    seen_names = set()

    for item in search_names:
        key = tuple(sorted(item.items()))
        if key not in seen_names:
            seen_names.add(key)
            unique_names.append(item)

    # Retrieve all available matching projects.
    # Search each name variant separately to avoid ambiguity about
    # how multiple PI-name criteria are combined by the API.
    projects_by_id = {}

    with requests.Session() as session:
        for search_name in unique_names:
            offset = 0

            while True:
                payload = {
                    "criteria": {
                        "pi_names": [search_name],
                        "include_active_projects": True
                    },
                    "offset": offset,
                    "limit": limit
                }

                response = session.post(
                    url,
                    json=payload,
                    timeout=30
                )
                response.raise_for_status()

                result = response.json()
                projects = result.get("results") or []

                for project in projects:
                    # Prefer stable application ID for deduplication.
                    project_id = (
                        project.get("appl_id")
                        or project.get("project_num")
                        or project.get("project_title")
                    )

                    if project_id is not None:
                        projects_by_id[str(project_id)] = project

                if len(projects) < limit:
                    break

                offset += limit

                # NIH documents a maximum offset of 14,999.
                if offset > 14999:
                    print(
                        "Warning: NIH API offset limit reached; "
                        "some results may not have been retrieved."
                    )
                    break

    projects_all = list(projects_by_id.values())

    # Name matching helpers.
    target_first = normalize(first)
    target_middle = normalize(middle)
    target_last = normalize(last)

    first_aliases = {
        target_first,
        *[
            normalize(alias)
            for alias in (name_aliases or [])
            if normalize(alias)
        ]
    }

    def first_name_matches(candidate):
        candidate = normalize(candidate)

        if not candidate:
            return False

        # Exact match or compatible initials.
        return any(
            candidate == alias
            or (
                len(candidate) == 1
                and alias.startswith(candidate)
            )
            or (
                len(alias) == 1
                and candidate.startswith(alias)
            )
            for alias in first_aliases
        )

    def investigator_name(investigator):
        parts = [
            clean(investigator.get("first_name")),
            clean(investigator.get("middle_name")),
            clean(investigator.get("last_name"))
        ]
        return " ".join(part for part in parts if part)

    
    records = []

    for project in projects_all:
        investigators = project.get("principal_investigators") or []

        matched = []

        for person in investigators:
            person_first = person.get("first_name")
            person_middle = normalize(person.get("middle_name"))
            person_last = normalize(person.get("last_name"))

            if not (
                person_last == target_last
                and first_name_matches(person_first)
            ):
                continue

            # Retain middle-name differences as a diagnostic.
            middle_status = (
                "Not supplied"
                if not target_middle or not person_middle
                else (
                    "Match"
                    if target_middle == person_middle
                    else (
                        "Initial-compatible"
                        if target_middle[0] == person_middle[0]
                        else "Different"
                    )
                )
            )

            matched.append({
                "name": investigator_name(person),
                "is_contact_pi": person.get("is_contact_pi"),
                "middle_status": middle_status,
                "profile_id": person.get("profile_id")
            })

        # Drop projects without exactly one matching investigator.
        if len(matched) != 1:
            continue

        person = matched[0]
        contact_status = person["is_contact_pi"]

        # Drop projects without a usable PI/contact-PI indicator.
        # Accept actual Boolean values only.
        if not isinstance(contact_status, bool):
            continue

        role = "PI" if contact_status else "Co-PI"

        # Drop projects without a usable matched name.
        if not person["name"]:
            continue

        records.append({
            "Matched Name": person["name"],
            "Role": role,
            "Participants": "; ".join(
                name for name in (
                    investigator_name(p)
                    for p in (project.get("principal_investigators") or [])
                )
                if name
            ),
            "Title": project.get("project_title"),
            "Abstract": project.get("abstract_text"),
            "Award Type": project.get("activity_code"),
            "Journal": pd.NA,
            "Amount": project.get("award_amount"),
            "NIH Email": pd.NA,
            "Institute": (
                (project.get("organization") or {}).get("org_name")
                or project.get("organization_name")
                or pd.NA
            ),
            "Start Date": project.get("project_start_date"),
            "End Date": project.get("project_end_date")
        })      
    
    df = pd.DataFrame(records)


    # Retain the original NSF-style researcher fields.
    df["First Name"] = first_name
    df["Middle Name"] = middle_name
    df["Last Name"] = last_name
    df["Searched Email"] = email

    output_columns = [
        "First Name",
        "Middle Name",
        "Last Name",
        "Participants",
        "Searched Email",
        "Matched Name",
        "Role",
        "Title",
        "Abstract",
        "Award Type",
        "Journal",
        "Amount",
        "NIH Email",
        "Institute",
        "Start Date",
        "End Date"
    ]

    for col in output_columns:
        if col not in df.columns:
            df[col] = pd.NA

    return df[output_columns]

In [5]:
df = pd.read_csv("scholar228_funding_triedby_junpeng.csv")[["identifier_first_name", "identifier_middle", "identifier_last_name", "identifier_email"]]

print(df)

    identifier_first_name identifier_middle identifier_last_name  \
0                Xiaojing               NaN                 Wang   
1                   Varun               NaN               Mishra   
2              Temiloluwa               NaN             Prioleau   
3                 Melissa               NaN                  Loh   
4          Yu Kuei (Alex)               NaN                  Lin   
..                    ...               ...                  ...   
222                   Bin               NaN                  Xie   
223               Mengjun               NaN                  Xie   
224                  Ting               NaN                  Zhu   
225                  Ingo               NaN               Vetter   
226                 Susan               NaN             Vorkoper   

            identifier_email  
0    xiaojing.wang@uconn.edu  
1      varun@varunmishra.com  
2    tprioleau@dartmouth.edu  
3       melissalkp@gmail.com  
4       yuklin@med.umich.edu

In [6]:
import time
import requests
import pandas as pd


def get_nih_awards_for_people(
    people,
    delay=0.2,
    rpp=25,
    name_aliases_by_email=None
):

    all_results = []

    for _, person in people.iterrows():

        first_name = person["identifier_first_name"]
        middle_name = person["identifier_middle"]
        last_name = person["identifier_last_name"]
        email = person["identifier_email"]

        print(f"\nProcessing: {first_name} {last_name}")

        try:
            name_aliases = (
                name_aliases_by_email.get(email, [])
                if name_aliases_by_email is not None
                else None
            )

            result = get_nih_awards(
                first_name=first_name,
                middle_name=middle_name,
                last_name=last_name,
                email=email,
                rpp=rpp,
                name_aliases=name_aliases
            )

            if not result.empty:
                all_results.append(result)

                print(
                    f"  Retrieved {len(result)} matching project(s)"
                )

                # Report records requiring identity or role review.
                if "Match Status" in result.columns:
                    needs_review = result["Match Status"].isin([
                        "No investigator match",
                        "Multiple investigator matches",
                        "Role unverified"
                    ]).sum()

                    if needs_review:
                        print(
                            f"  Warning: {needs_review} project(s) "
                            "need investigator/role verification"
                        )
            else:
                print("  No matching projects found")

        except requests.exceptions.RequestException as e:
            print(
                f"  API error for {first_name} {last_name}: {e}"
            )

        except Exception as e:
            print(
                f"  Error for {first_name} {last_name}: {e}"
            )

        time.sleep(delay)

    if not all_results:
        return pd.DataFrame(columns=[
            "First Name",
            "Middle Name",
            "Last Name",
            "Searched Email",
            "Match Name",
            "Role",
            "Participants",
            "Title",
            "Abstract",
            "Award Type",
            "Journal",
            "Amount",
            "NIH Email",
            "Institute",
            "Start Date",
            "End Date"
        ])

    return pd.concat(all_results, ignore_index=True)

In [7]:
results = get_nih_awards_for_people(df)


Processing: Xiaojing Wang
  Retrieved 1 matching project(s)

Processing: Varun Mishra
  No matching projects found

Processing: Temiloluwa Prioleau
  No matching projects found

Processing: Melissa Loh
  No matching projects found

Processing: Yu Kuei (Alex) Lin
  No matching projects found

Processing: Shanthini Kasturi
  No matching projects found

Processing: Carolyn Yeager
  No matching projects found

Processing: Justin Knox
  Retrieved 5 matching project(s)

Processing: Kara Christensen
  No matching projects found

Processing: Laura Bellows
  Retrieved 1 matching project(s)

Processing: Heather Nuske
  Retrieved 1 matching project(s)

Processing: Caroline Silva
  Retrieved 1 matching project(s)

Processing: Caroline Dorfman
  Retrieved 1 matching project(s)

Processing: Mary Ellen Vajravelu
  No matching projects found

Processing: Jessica Lipschitz
  No matching projects found

Processing: Agatha Lenartowicz
  Retrieved 4 matching project(s)

Processing: Jeff Epstein
  Retriev

In [8]:
results["Role"].unique()

array(['PI', 'Co-PI'], dtype=object)

In [9]:
results

,First Name,Middle Name,Last Name,Participants,Searched Email,Matched Name,Role,Title,Abstract,Award Type,Journal,Amount,NIH Email,Institute,Start Date,End Date
0,Xiaojing,NaN,Wang,Xiaojing Wang,xiaojing.wang@uconn.edu,Xiaojing Wang,PI,Integrating long-read and short-read RNA seque...,PROJECT SUMMARY\nHeart failure (HF) is a compl...,R21,<NA>,118125,<NA>,UNIVERSITY OF TEXAS HLTH SCIENCE CENTER,2026-08-19T00:00:00,2028-06-30T00:00:00
1,Justin,NaN,Knox,Sarah K Keedy; Justin Knox; John Schneider,justinryanknox@gmail.com,Justin Knox,Co-PI,"Heavy cannabis use, neurocognition and PrEP ca...",We propose to conduct research that will deter...,R01,<NA>,818585,<NA>,UNIVERSITY OF CHICAGO,2022-09-15T00:00:00,2027-06-30T00:00:00
2,Justin,NaN,Knox,Justin Knox; Jade Pagkas-Bather,justinryanknox@gmail.com,Justin Knox,PI,"Informing, Updating and Adapting an Existing E...",Project Summary\nAdvancements in biomedical st...,R21,<NA>,466466,<NA>,NEW YORK STATE PSYCHIATRIC INSTITUTE DBA RESEA...,2026-08-01T00:00:00,2028-07-31T00:00:00
3,Justin,NaN,Knox,Dustin T Duncan; Justin Knox,justinryanknox@gmail.com,Justin Knox,Co-PI,"Cannabis use, PrEP and HIV transmission risk",Populations with elevated risk for HIV face a ...,R01,<NA>,709417,<NA>,COLUMBIA UNIVERSITY HEALTH SCIENCES,2021-08-01T00:00:00,2027-05-31T00:00:00
4,Justin,NaN,Knox,Justin Knox,justinryanknox@gmail.com,Justin Knox,PI,Intervening to improve HIV treatment and reduc...,Significance. For many people living with HIV ...,K01,<NA>,142597,<NA>,NEW YORK STATE PSYCHIATRIC INSTITUTE DBA RESEA...,2020-09-01T00:00:00,2027-08-31T00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,Parya,NaN,Saberi,Parya Saberi,NaN,Parya Saberi,PI,Development Core,PROJECT SUMMARY/ABSTRACT – Development and Men...,P30,<NA>,314076,<NA>,"UNIVERSITY OF CALIFORNIA, SAN FRANCISCO",2026-09-01T00:00:00,2031-08-31T00:00:00
195,Parya,NaN,Saberi,Parya Saberi,NaN,Parya Saberi,PI,Mentoring Clinical Investigators in Psychedeli...,Project Summary/Abstract\nResearch on psychede...,K24,<NA>,216634,<NA>,"UNIVERSITY OF CALIFORNIA, SAN FRANCISCO",2025-08-01T00:00:00,2030-07-31T00:00:00
196,Parya,NaN,Saberi,Parya Saberi,NaN,Parya Saberi,PI,"Comprehensive, tailored, technology-based inte...","Abstract/Project Summary:\nIn the US, youth an...",R01,<NA>,608783,<NA>,"UNIVERSITY OF CALIFORNIA, SAN FRANCISCO",2022-09-01T00:00:00,2027-06-30T00:00:00
197,Johannes,NaN,Thrul,Johannes Thrul; RYAN G VANDREY,NaN,Johannes Thrul,Co-PI,Development of a Longitudinal Observational Re...,"Due to changes in cannabis policy, most Americ...",UM1,<NA>,2001470,<NA>,JOHNS HOPKINS UNIVERSITY,2023-06-01T00:00:00,2028-04-30T00:00:00


In [10]:
results.to_excel("NIH_raw.xlsx")

## Merge with Demographic Data

In [11]:
import pandas as pd

df_nsf = pd.read_excel("NIH_raw.xlsx")
df_demo = pd.read_csv("scholar176_with_demographic.csv")

In [12]:
df_nsf2 = df_nsf.copy()
df_demo2 = df_demo.copy()

df_demo2["name_clean"] = (
    df_demo2["name"]
    .astype("string")
    .str.strip()
    # Remove Dr. / Dr at the beginning
    .str.replace(r"^\s*Dr\.?\s+", "", regex=True, case=False)
    # Remove one-letter middle-name entries such as M or M.
    .str.replace(r"\s+[A-Za-z]\.?(?=\s+[A-Za-z])", "", regex=True)
    # Remove one-letter middle-name entries such as M or M., including at the beginning
    .str.replace(r"(?:^|\s+)[A-Za-z]\.?(?=\s+[A-Za-z])", "", regex=True)
    # Remove anything inside parentheses, including the parentheses
    .str.replace(r"\([^)]*\)", "", regex=True)
    # Remove everything after the first comma, including the comma
    .str.replace(r",.*$", "", regex=True)
    .str.strip()
)

# Split the cleaned name
name_parts = df_demo2["name_clean"].str.split()

df_demo2["_first_match"] = name_parts.str[0]
df_demo2["_last_match"] = name_parts.str[-1]


# --- Create matching keys for df_nsf ---

df_nsf2["_first_match"] = (
    df_nsf2["First Name"]
    .astype("string")
    .str.strip()
    .str.split()
    .str[0]
)

df_nsf2["_last_match"] = (
    df_nsf2["Last Name"]
    .astype("string")
    .str.strip()
    .str.split()
    .str[-1]
)


# --- Normalize capitalization for matching ---

df_demo2["_first_match"] = df_demo2["_first_match"].str.lower()
df_demo2["_last_match"] = df_demo2["_last_match"].str.lower()

df_nsf2["_first_match"] = df_nsf2["_first_match"].str.lower()
df_nsf2["_last_match"] = df_nsf2["_last_match"].str.lower()


# --- Information to add from df_demo ---

demo_columns = [
    "record_id",
    "name",
    "Cohort",
    "gender",
    "stage",
    "com_discipline",
    "region"
]

demo_info = df_demo2[
    ["_first_match", "_last_match"] + demo_columns
]


# --- Merge onto NSF data ---

df_nsf2 = df_nsf2.merge(
    demo_info,
    on=["_first_match", "_last_match"],
    how="left"
)


# --- Remove temporary matching columns ---

df_nsf2 = df_nsf2.drop(
    columns=["_first_match", "_last_match"]
)

In [13]:
df_nsf2

,Unnamed: 0,First Name,Middle Name,Last Name,Participants,Searched Email,Matched Name,Role,Title,Abstract,...,Institute,Start Date,End Date,record_id,name,Cohort,gender,stage,com_discipline,region
0,0,Xiaojing,NaN,Wang,Xiaojing Wang,xiaojing.wang@uconn.edu,Xiaojing Wang,PI,Integrating long-read and short-read RNA seque...,PROJECT SUMMARY\nHeart failure (HF) is a compl...,...,UNIVERSITY OF TEXAS HLTH SCIENCE CENTER,2026-08-19T00:00:00,2028-06-30T00:00:00,1.0,Xiaojing Wang,2022.0,male,mid-career,CS / Stats / Engineering / Data Science,Northeast
1,1,Justin,NaN,Knox,Sarah K Keedy; Justin Knox; John Schneider,justinryanknox@gmail.com,Justin Knox,Co-PI,"Heavy cannabis use, neurocognition and PrEP ca...",We propose to conduct research that will deter...,...,UNIVERSITY OF CHICAGO,2022-09-15T00:00:00,2027-06-30T00:00:00,8.0,Justin Knox,2022.0,female,mid-career,Public Health,Northeast
2,2,Justin,NaN,Knox,Justin Knox; Jade Pagkas-Bather,justinryanknox@gmail.com,Justin Knox,PI,"Informing, Updating and Adapting an Existing E...",Project Summary\nAdvancements in biomedical st...,...,NEW YORK STATE PSYCHIATRIC INSTITUTE DBA RESEA...,2026-08-01T00:00:00,2028-07-31T00:00:00,8.0,Justin Knox,2022.0,female,mid-career,Public Health,Northeast
3,3,Justin,NaN,Knox,Dustin T Duncan; Justin Knox,justinryanknox@gmail.com,Justin Knox,Co-PI,"Cannabis use, PrEP and HIV transmission risk",Populations with elevated risk for HIV face a ...,...,COLUMBIA UNIVERSITY HEALTH SCIENCES,2021-08-01T00:00:00,2027-05-31T00:00:00,8.0,Justin Knox,2022.0,female,mid-career,Public Health,Northeast
4,4,Justin,NaN,Knox,Justin Knox,justinryanknox@gmail.com,Justin Knox,PI,Intervening to improve HIV treatment and reduc...,Significance. For many people living with HIV ...,...,NEW YORK STATE PSYCHIATRIC INSTITUTE DBA RESEA...,2020-09-01T00:00:00,2027-08-31T00:00:00,8.0,Justin Knox,2022.0,female,mid-career,Public Health,Northeast
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,194,Parya,NaN,Saberi,Parya Saberi,NaN,Parya Saberi,PI,Development Core,PROJECT SUMMARY/ABSTRACT – Development and Men...,...,"UNIVERSITY OF CALIFORNIA, SAN FRANCISCO",2026-09-01T00:00:00,2031-08-31T00:00:00,221.0,Parya Saberi,2015.0,male,mid-career,Psychology,West
195,195,Parya,NaN,Saberi,Parya Saberi,NaN,Parya Saberi,PI,Mentoring Clinical Investigators in Psychedeli...,Project Summary/Abstract\nResearch on psychede...,...,"UNIVERSITY OF CALIFORNIA, SAN FRANCISCO",2025-08-01T00:00:00,2030-07-31T00:00:00,221.0,Parya Saberi,2015.0,male,mid-career,Psychology,West
196,196,Parya,NaN,Saberi,Parya Saberi,NaN,Parya Saberi,PI,"Comprehensive, tailored, technology-based inte...","Abstract/Project Summary:\nIn the US, youth an...",...,"UNIVERSITY OF CALIFORNIA, SAN FRANCISCO",2022-09-01T00:00:00,2027-06-30T00:00:00,221.0,Parya Saberi,2015.0,male,mid-career,Psychology,West
197,197,Johannes,NaN,Thrul,Johannes Thrul; RYAN G VANDREY,NaN,Johannes Thrul,Co-PI,Development of a Longitudinal Observational Re...,"Due to changes in cannabis policy, most Americ...",...,JOHNS HOPKINS UNIVERSITY,2023-06-01T00:00:00,2028-04-30T00:00:00,223.0,Johannes Thrul,2015.0,female,early-career,Psychology,West


In [14]:
df_nsf2.to_excel("NIH_demographic.xlsx", index=False)